<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/BT_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bluetooth Security Mechanisms Overview

Bluetooth security has evolved significantly across different versions, particularly with the transition from Bluetooth Classic (BR/EDR) to Bluetooth Low Energy (BLE). Below is an overview of the primary security mechanisms built into the Bluetooth standard.

---

### 1. Pairing and Association Models
Pairing is the initial process where two devices establish a shared secret (Link Key in Classic, Long Term Key in BLE) to secure future communications.

*   **Legacy Pairing (Bluetooth 2.0 & older):** Relied on a static PIN code. It is highly vulnerable to passive eavesdropping during the pairing phase.
*   **Secure Simple Pairing (SSP) / LE Secure Connections:** Introduced in Bluetooth 2.1 (Classic) and Bluetooth 4.2 (BLE), this uses **Elliptic Curve Diffie-Hellman (ECDH)** key exchange to protect against passive eavesdropping.
    *   **Just Works:** Designed for devices without displays or keyboards. It provides no protection against Man-in-the-Middle (MITM) attacks.
    *   **Numeric Comparison:** Both devices display a 6-digit number. The user confirms they match, protecting against MITM attacks.
    *   **Passkey Entry:** One device displays a random 6-digit passkey, and the user enters it on the other device.
    *   **Out of Band (OOB):** Uses an external communication channel (such as NFC) to exchange discovery and cryptographic details.

### 2. Encryption and Data Integrity
Once pairing is complete, all data transmitted between the devices is encrypted and verified for integrity.

*   **Encryption Algorithms:** Modern Bluetooth (Secure Connections) utilizes strong symmetric encryption standards:
    *   **AES-CCM (128-bit):** Used in BLE to provide both confidentiality (encryption) and data authenticity.
    *   **AES-GCM / HMAC:** Used in some configurations to verify that data has not been altered in transit.
*   **Key Size Negotiation:** To prevent downgrade attacks where an attacker forces the use of weak encryption, modern specifications enforce a minimum key entropy of 7 bytes (and recommend 16 bytes/128-bit).

### 3. Device Privacy and Address Resolution
To prevent malicious actors from tracking a Bluetooth device over time using its hardware MAC address, Bluetooth introduces privacy features:

*   **Resolvable Private Addresses (RPA):** The device periodically changes its broadcasted MAC address (typically every 15 minutes).
*   **Identity Resolving Key (IRK):** Trusted bonded devices share an IRK during the initial pairing phase. This allows authorized peer devices to resolve the changing random MAC address back to the static identity, while untrusted scanners see only changing, unlinkable addresses.

# Bluetooth Legacy Pairing Vulnerabilities

Bluetooth Legacy Pairing (used in Bluetooth 2.0 and earlier, and still supported as a fallback in many modern devices) suffers from several design flaws that make it highly insecure against active and passive attackers.

### 1. Passive Eavesdropping and Key Recovery
In Legacy Pairing, the security of the entire connection relies on a temporary key ($K_{init}$) derived from three publicly visible components and a user-entered PIN code:

$$\text{Security Depends On:} \quad \text{PIN} + \text{BD\_ADDR}_{master} + \text{BD\_ADDR}_{slave} + \text{Random Number } (IN\_RAND)$$

*   **The Flaw:** The Master and Slave Bluetooth Device Addresses (`BD_ADDR`) are transmitted in plaintext over the air. The random number `IN_RAND` is also sent in plaintext.
*   **The Attack:** If an attacker captures the initial pairing handshake (specifically the exchange of the random challenges and the commitment values), the only unknown variable is the **PIN code**.

### 2. PIN Brute-Forcing (The crackle Attack)
Because the PIN code is often short (typically a 4-digit numeric code like `0000` or `1234` for headless devices like headsets), it has very low entropy.

*   An attacker who sniffs the initial connection setup can easily run an offline brute-force search against all possible PIN values ($10,000$ combinations for a 4-digit PIN) in less than a second on standard hardware.
*   Once the correct PIN is identified, the attacker can derive the Link Key ($K_{link}$) and decrypt all subsequent traffic or impersonate the target device.

### 3. Lack of Forward Secrecy
Legacy Pairing does not use an ephemeral key exchange protocol like Diffie-Hellman (ECDH).

*   If an attacker logs encrypted traffic today and successfully intercepts or brute-forces the PIN/Link Key at any point in the future, they can decrypt all historical sessions they previously recorded.

# Comparison: Legacy Pairing vs. Secure Simple Pairing (SSP)

| Feature | Legacy Pairing (Bluetooth v2.0 & older) | Secure Simple Pairing (SSP) (Bluetooth v2.1 & newer) |
| :--- | :--- | :--- |
| **Key Exchange Protocol** | Symmetric derivation based on a shared PIN | Elliptic Curve Diffie-Hellman (ECDH) |
| **Symmetric Key Strength** | Weak (derived directly from low-entropy PIN) | Strong (128-bit Link Key derived via ECDH public-private keys) |
| **Passive Eavesdropping Protection** | **None.** Anyone sniffing the air during pairing can compute the Link Key if they know or brute-force the PIN. | **Highly Secure.** An eavesdropper grabbing all over-the-air packets cannot derive the shared key due to ECDH properties. |
| **Man-in-the-Middle (MITM) Protection** | Basic (only as strong as the user's ability to keep the PIN secret) | Strong (uses Numeric Comparison, Passkey Entry, or Out of Band verification) |
| **User Interaction Models** | Static PIN entry (often hardcoded, e.g., `0000`) | Multiple association models tailored to device IO capabilities (Just Works, Numeric Comparison, etc.) |

### Key Improvements in SSP

1. **ECDH Cryptography:** SSP uses Diffie-Hellman key exchange, which allows two devices to establish a shared secret over an insecure channel without actually sending the secret or easily crackable components of it over the air.
2. **Prevention of Offline Brute-Force Attacks:** Even if a malicious actor records the entire pairing process, they cannot brute-force the link key offline because they lack the private keys generated locally on each device.
3. **Granular Association Models:** SSP dynamically negotiates the pairing method based on the Input/Output capabilities of both devices, ensuring the highest level of security possible for that combination of hardware.

# Executive Summary: Key Security Shifts from Legacy to SSP

When transitioning from **Legacy Pairing** to **Secure Simple Pairing (SSP)**, the standard introduced fundamental shifts in threat mitigation:

1. **Cryptographic Foundation:** Legacy pairing relies on a shared PIN that is directly vulnerable to passive over-the-air capturing. SSP implements **Elliptic Curve Diffie-Hellman (ECDH)**, meaning eavesdroppers cannot calculate the encryption key even if they monitor the entire pairing process.
2. **Brute-Force Resistance:** Legacy pairing can be broken offline via automated tools in milliseconds due to low-entropy PINs (e.g., 4 digits). SSP defeats offline brute-force attacks by keeping the private parameters of key generation secure on the local devices.
3. **Active Attack (MITM) Defense:** While Legacy pairing has weak, static methods, SSP utilizes dynamic, context-aware verification models (such as *Numeric Comparison* and *Passkey Entry*) to ensure that a third party is not actively intercepting or altering the communication channel during pairing.

# BLE Just Works Pairing: Security Trade-offs

**Just Works** is one of the standard association models in Bluetooth Low Energy (BLE) pairing. It is widely used but presents a distinct set of trade-offs between convenience and security.

### What is "Just Works"?
Just Works is mathematically identical to the **Numeric Comparison** pairing model. It uses Elliptic Curve Diffie-Hellman (ECDH) to establish a secure, encrypted link. The key difference is that Just Works is designed for devices with **No Input No Output (NoIO)** capabilities (such as fitness trackers, smart bulbs, or basic sensors).

Instead of displaying a 6-digit number to the user to manually confirm, the confirmation step is skipped, and the protocol automatically assumes a match.

---

### The Security Trade-offs

#### Pros (The Benefits)
*   **User Convenience:** Pairing requires no human intervention, password typing, or code validation, creating a seamless user experience.
*   **Hardware Cost & Design:** Allows product manufacturers to build tiny, low-power devices without needing screens, buttons, or keyboards.
*   **Passive Eavesdropping Protection:** Because it uses ECDH, a passive attacker sitting nearby and recording the radio signals during pairing cannot decode the encryption keys.

#### Cons (The Risks)
*   **No Protection Against Man-in-the-Middle (MITM) Attacks:** Since there is no out-of-band or human verification step, a nearby attacker can intercept the key exchange. The attacker places themselves between the two devices, establishing separate secure connections with both. To the devices, everything "Just Works," but the attacker can inspect, modify, or inject data.
*   **Inability to Authenticate the Peer:** There is no cryptographic guarantee that the phone is pairing with the actual device in front of the user rather than an identical device in an adjacent room or a malicious simulator.